# 模仿学习简介

从专家示教中学习策略方法，无需手动设置奖励函数（奖励函数难定义）。

# 监督学习引入（行为克隆）

将专家的状态观测作为输入（X），将专家的动作为标签（y），然后训练一个分类或回归模型。随着机器学习的发展，机器人开始具备一定的泛化能力。

由于模型不完美，机器人会犯小错误，导致进入专家从未演示过的状态。此时，模型不知道该怎么办，错误会像滚雪球一样越积越大，最终导致任务失败。这个问题被称为分布偏移。

**这意味着 BC 在短时任务上表现尚可，但在长时任务上容易崩溃。**

## DAgger

执行任务时，如果进入不确定的状态，就向专家求助。

专家给出一个新的 (状态，动作) 对，加入训练集。

局限性：需要专家标注。

## 逆强化学习（IRL）

逆强化学习（IRL）从专家示教中反推奖励函数，再用 RL 算法基于学到的奖励训练策略。

计算复杂度高：正逆RL都要更新。

# 深度学习方法

## 生成对抗模仿学习（GAIL）

- 生成器：智能体策略，试图产生与专家行为无法区分的轨迹

- 判别器：区分专家与生成器轨迹

比行为克隆更鲁棒，比逆强化学习的计算效率更高。

# 大模型方法

## 4.1 模仿学习 + Transformers / 大模型 (Vision-Language-Action Models, VLA)

这种方法的核心思想是：**将机器人控制问题转化为一个大规模的多模态序列预测问题**，直接利用互联网规模预训练的大模型（LLM/VLM）的“世界知识”和强大的表征能力，实现端到端的策略学习。

### 1. 工作机制
* **多模态 Token 化**：将机器人的视觉输入（图像/视频）、自然语言指令、本体感觉（关节角度、末端位姿）以及历史动作，全部离散化为统一的 Token 序列。
* **自回归预测**：使用标准的 Transformer 架构（Decoder-only 或 Encoder-Decoder），以历史状态和语言指令为条件，自回归地预测下一个动作 Token（或动作块 Action Chunk）。
* **大规模联合微调**：收集成百上千台机器人在不同场景、执行不同任务的海量演示数据，对预训练好的视觉-语言模型（如 PaLM、ViT）进行全参数或高效微调（如 LoRA），使其输出从“文本/图像”转变为“机器人动作”。

### 2. 核心优势
* **零样本/少样本组合泛化**：这是最大的突破。如 RT-2 所示，模型不仅能执行训练过的任务，还能理解全新的、组合性的抽象指令（例如“把那个快过期的苹果扔掉”），因为它继承了预训练模型的语义理解和常识推理能力。
* **统一的多模态表征**：打破了传统机器人系统中“感知-规划-控制”的模块化壁垒，视觉、语言和动作在同一个高维空间中对齐，减少了信息传递过程中的误差累积。
* **开箱即用的鲁棒性**：大模型对光照变化、背景 clutter（杂乱）、物体遮挡等具有天然的鲁棒性。

### 3. 代表性工作
* **Google RT-1 / RT-2**：RT-1 证明了 Transformer 在大规模机器人数据上的可扩展性；RT-2 则进一步将模型与视觉-语言模型（VLM）对齐，诞生了 VLA（Vision-Language-Action）模型范式。
* **OpenVLA / Octo**：开源社区推出的视觉-语言-动作模型，旨在降低 VLA 的训练门槛，并证明在多样化开源数据集上训练的模型具有强大的跨任务泛化能力。

### 4. 面临的挑战
* **推理延迟**：Transformer 的自回归生成机制计算量大，可能无法满足机器人高频（如 50Hz-100Hz）的实时控制需求（通常需借助 Action Chunking 动作分块技术缓解）。
* **计算与数据成本**：训练和微调百亿参数级别的模型需要庞大的算力集群和高质量的百万级机器人轨迹数据。
* **幻觉问题**：大模型可能会生成看似合理但物理上不可行或危险的动作。

---

## 4.2 模仿学习 + 自监督学习 (Self-Supervised Learning for Robotics)

这种方法的核心思想是：**“预训练 + 微调”范式在机器人领域的延伸**。既然高质量的“状态-动作”专家对极其昂贵，那就先用海量的、廉价的**无标注数据**让模型学会“理解世界”和“理解动作”，然后再用极少量的专家数据“点拨”它完成具体任务。

### 1. 工作机制
* **阶段一：自监督预训练（学习先验）**
  * **数据源**：海量的互联网人类视频（如 YouTube 上的做饭、组装视频）、或机器人无目标探索产生的无标签数据。
  * **预训练任务**：
    * *表征学习*：如对比学习（R3M, VIP），让模型学会提取与任务相关的、对视角和背景变化不变的视觉特征。
    * *世界模型*：如掩码视频重建（Masked Video Modeling）或下一帧预测，让模型在隐空间中学习物理规律（如重力、物体恒常性、碰撞）。
    * *行为先验*：预测视频中人物的手部边界框或未来动作意图。
* **阶段二：监督微调（模仿学习）**
  * 冻结或部分解冻预训练模型的权重，将其作为特征提取器或初始化策略。
  * 使用少量（几十到几百条）特定任务的专家演示数据，通过行为克隆（BC）或强化学习（RL）进行快速微调，对齐到具体的机器人动作空间。

#### <mark>世界模型</mark>
一个机器学习系统，它能够构建出对外部环境的内部表征（Internal Representation），并能够预测在给定当前状态和智能体（Agent）动作的情况下，环境未来将如何演化。

核心组件：
- 感知压缩：将高维的原始输入（如摄像头拍摄的像素图像）压缩成一个低维的、包含关键语义信息的“潜变量”（Latent Vector）。模型不需要记住背景的每一片树叶，只需记住“那里有一个红色的杯子”。

- 动态预测：这是世界模型的核心（通常使用 RNN、Transformer 或状态空间模型）。它接收当前的潜变量和智能体打算采取的动作，预测下一个时间步的潜变量。

- 价值/策略评估：智能体在这个“内部模拟”的潜空间中尝试各种动作序列（想象未来），评估哪种动作能带来最高的奖励，然后将最优策略应用到真实世界中。

优点：
- 减少试错成本：模拟试错
- 从无标注数据中学习知识
- 具备长期规划能力：模拟试错，评估多步行动

### 2. 核心优势
* **打破数据瓶颈**：将机器人学习对“昂贵机器人演示数据”的依赖，转移到了“廉价且无限的互联网视频数据”上，极大提升了样本效率（Sample Efficiency）。
* **跨域常识注入**：人类视频中蕴含了丰富的物理交互常识（如“杯子装满水后不能倾斜”），这些常识通过自监督学习被编码到模型中，弥补了机器人真实交互数据的不足。
* **更优的特征空间**：自监督学习到的表征通常比直接从随机初始化开始训练的表征更平滑、更具泛化性，能有效缓解模仿学习中的协变量偏移（Covariate Shift）问题。

### 3. 代表性工作
* **R3M / VIP / MVP**：使用大规模人类动作视频，通过时间对比学习或距离预测，预训练出适用于多种机器人操作任务的通用视觉表征。
* **PerAct / 3D Diffusion Policy**：结合 3D 视觉表征与扩散模型，利用自监督学到的 3D 几何先验，仅需少量演示即可实现高精度的 6-DoF 操作。

### 4. 面临的挑战
* **域鸿沟（Domain Gap）**：互联网视频通常是第三人称视角、人类形态（双手），而机器人是第一人称视角、机械臂形态。如何将人类视频的“语义知识”有效映射到机器人的“运动学空间”是一个巨大难题。
* **表征与任务的对齐**：自监督学习的目标（如预测下一帧）与下游控制任务的目标（如成功抓取）并不完全一致，可能导致学到的特征对最终任务帮助有限（任务无关特征）。

---

## 总结与对比

| 维度 | 4.1 模仿学习 + Transformers/大模型 | 4.2 模仿学习 + 自监督学习 |
| :--- | :--- | :--- |
| **核心驱动力** | **规模效应** (Scaling Law)：用大模型、大数据换取泛化能力。 | **数据效率** (Data Efficiency)：用无监督预训练降低对标注数据的依赖。 |
| **知识来源** | 互联网规模的图文多模态预训练模型 + 大规模机器人微调数据。 | 海量无标注视频（人类或机器人） + 少量专家演示数据。 |
| **主要解决问题** | 策略的语义理解、组合泛化、开放词汇（Open-vocabulary）任务。 | 专家数据获取成本高、冷启动困难、底层物理常识缺失。 |
| **计算重心** | 推理和微调阶段的算力消耗巨大（大模型前向传播）。 | 预训练阶段需要大量算力，但下游微调阶段非常轻量。 |

**未来趋势：两者的融合**

目前，这两个方向正在快速交汇。最前沿的研究（如某些最新的 VLA 模型）已经开始**使用自监督学习的方法来预训练视觉-语言-动作大模型**。例如，先让大模型在海量无标注的机器人视频上进行掩码重建或对比学习，学习通用的物理和动作先验，然后再用多任务专家数据进行指令微调。这种结合有望同时获得“大模型的泛化能力”和“自监督学习的数据高效性”。

# 现代模仿学习方法

## Diffusion Policy

将动作生成转化为一个条件去噪过程：
- 它先预测一个充满噪声的动作序列。
- 然后根据当前的视觉观测和任务条件，一步步“去噪”，最终还原出一个符合物理规律、且能完美匹配专家演示多模态分布的平滑动作序列。
- 结果：它在复杂的接触丰富任务（如衣物折叠、线缆插拔、推挤操作）中，成功率大幅超越了传统 BC 和 GAIL，成为了当前机器人底层控制策略的 SOTA 基线。
